<a href="https://colab.research.google.com/github/sharvani1357/Positional_Encoding/blob/main/Positional_Encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Positional Encoding and self Attention


In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras. layers import TextVectorization, Embedding, MultiHeadAttention

Input Sentence


In [ ]:
sentence=["I love Deep Learning"]
print(sentence)

['I love Deep Learning']


Tokenization


In [ ]:
vectorizer=TextVectorization(output_mode="int",output_sequence_length=4)
vectorizer.adapt(sentence)

tokens=vectorizer(sentence)

print("Vocabulary:")
print(vectorizer.get_vocabulary())

print("Tokens:")
print(tokens.numpy())

Vocabulary:
['', '[UNK]', np.str_('love'), np.str_('learning'), np.str_('i'), np.str_('deep')]
Tokens:
[[4 2 5 3]]


Word Embeddings

In [ ]:
embedding_dim=8

embedding_layer=Embedding(input_dim=len(vectorizer.get_vocabulary()),output_dim=embedding_dim)
word_embeddings=embedding_layer(tokens)

print("Word Embeddings:")
print(word_embeddings.numpy())

Word Embeddings:
[[[-0.04289773 -0.02708997  0.00757289 -0.04751365 -0.0025358
    0.02450963 -0.03576667 -0.01732689]
  [-0.02676149  0.04024203  0.01558315  0.04569456 -0.00661621
   -0.03813095 -0.03885699  0.01538581]
  [-0.0044893   0.01570424  0.03151596 -0.04929818 -0.03245219
   -0.00870303  0.04240141 -0.00575209]
  [ 0.00547075 -0.00448249 -0.0315569   0.02870176  0.02903268
    0.01106773 -0.01861111  0.0279601 ]]]


Positional Encoding Function

In [ ]:
def positional_encoding(max_position, d_model):
    positions = np.arange(max_position)[:, np.newaxis]
    dimensions = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(
        10000,
        (2 * (dimensions // 2)) / np.float32(d_model)
    )

    angle_rads = positions * angle_rates

    PE = np.zeros((max_position, d_model))

    PE[:, 0::2] = np.sin(angle_rads[:, 0::2])
    PE[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(PE, dtype=tf.float32)

In [ ]:
PE=positional_encoding(4,embedding_dim)
print(PE.numpy())

[[ 0.0000000e+00  1.0000000e+00  0.0000000e+00  1.0000000e+00
   0.0000000e+00  1.0000000e+00  0.0000000e+00  1.0000000e+00]
 [ 8.4147096e-01  5.4030228e-01  9.9833414e-02  9.9500418e-01
   9.9998331e-03  9.9994999e-01  9.9999981e-04  9.9999952e-01]
 [ 9.0929741e-01 -4.1614684e-01  1.9866933e-01  9.8006660e-01
   1.9998666e-02  9.9980003e-01  1.9999987e-03  9.9999797e-01]
 [ 1.4112000e-01 -9.8999250e-01  2.9552022e-01  9.5533651e-01
   2.9995501e-02  9.9955004e-01  2.9999956e-03  9.9999553e-01]]


Add Positional Encoding

In [ ]:
position_aware_embeddings = word_embeddings + PE[tf.newaxis, :]

print("Position-aware Embeddings:")
print(position_aware_embeddings.numpy())

Position-aware Embeddings:
[[[-0.04289773  0.97291005  0.00757289  0.95248634 -0.0025358
    1.0245097  -0.03576667  0.9826731 ]
  [ 0.8147095   0.5805443   0.11541656  1.0406988   0.00338362
    0.96181905 -0.037857    1.0153854 ]
  [ 0.9048081  -0.4004426   0.23018529  0.93076843 -0.01245352
    0.991097    0.04440141  0.9942459 ]
  [ 0.14659075 -0.994475    0.2639633   0.9840383   0.05902818
    1.0106177  -0.01561111  1.0279557 ]]]


Multi-Head Attention

In [ ]:
attention_layer = MultiHeadAttention(
    num_heads=2,
    key_dim=embedding_dim
)

In [ ]:
# Apply Self Attention
attention_output = attention_layer(
    query=position_aware_embeddings,
    value=position_aware_embeddings,
    key=position_aware_embeddings,
)

print(attention_output.shape)
print("Contextualized Embeddings:")
print(attention_output.numpy())

(1, 4, 8)
Contextualized Embeddings:
[[[-0.02452116 -0.03721277  0.14848232 -0.5719228   0.580593
    0.19715458 -0.10106492  0.17350365]
  [-0.0222843  -0.03776588  0.15098733 -0.57142043  0.58116496
    0.19331688 -0.09686518  0.16969204]
  [-0.01840818 -0.03935364  0.14847238 -0.571725    0.578214
    0.19168495 -0.09508616  0.16670594]
  [-0.01836908 -0.03971331  0.14525627 -0.57224333  0.5765747
    0.19420488 -0.09747783  0.16805938]]]
